In [1]:
"""Módulo para unificar todos los JSON de data/parsed_json en un único DataFrame."""

from __future__ import annotations

import json
import logging
from pathlib import Path

import pandas as pd

logger = logging.getLogger(__name__)


def _resolve_default_dir() -> Path:
    """Resuelve data/parsed_json sin depender de __file__ (compatible con Notebook)."""
    # 1) Relativo al cwd del notebook
    candidate = Path("data") / "parsed_json"
    if candidate.is_dir():
        return candidate.resolve()

    # 2) Si existe script_path (ya definido en el notebook)
    if "script_path" in globals():
        p = Path(script_path)
        return (p.parents[2] / "data" / "parsed_json").resolve()

    # 3) Si existe path (ya definido en el notebook)
    if "path" in globals():
        p = Path(path)
        return (p.parents[2] / "data" / "parsed_json").resolve()

    return candidate.resolve()


_DEFAULT_DIR = _resolve_default_dir()


def load_articles_df(parsed_dir: Path | str | None = None) -> pd.DataFrame:
    """Lee todos los archivos .json de *parsed_dir* y retorna un DataFrame unificado."""
    parsed_dir = Path(parsed_dir) if parsed_dir else _DEFAULT_DIR

    if not parsed_dir.is_dir():
        raise FileNotFoundError(f"Directorio no encontrado: {parsed_dir}")

    frames: list[pd.DataFrame] = []

    for json_file in sorted(parsed_dir.glob("*.json")):
        try:
            with open(json_file, encoding="utf-8") as fh:
                data = json.load(fh)
        except (json.JSONDecodeError, OSError) as exc:
            logger.warning("No se pudo leer %s: %s", json_file.name, exc)
            continue

        if not data:
            logger.info("Archivo vacío: %s", json_file.name)
            continue

        df = pd.DataFrame(data)
        df["source_file"] = json_file.stem
        frames.append(df)
        logger.info("Cargado %s — %d registros", json_file.name, len(df))

    if not frames:
        logger.warning("No se encontraron datos en %s", parsed_dir)
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True)

    for col_src, col_dst in [
        ("fecha_publicacion", "fecha_publicacion_dt"),
        ("fecha_obtencion", "fecha_obtencion_dt"),
    ]:
        if col_src in combined.columns:
            combined[col_dst] = pd.to_datetime(
                combined[col_src], format="%d/%m/%Y", errors="coerce"
            )

    if "url_noticia" in combined.columns:
        before = len(combined)
        combined.drop_duplicates(subset=["url_noticia"], keep="first", inplace=True)
        dupes = before - len(combined)
        if dupes:
            logger.info("Eliminados %d duplicados por url_noticia", dupes)

    if "fecha_publicacion_dt" in combined.columns:
        combined.sort_values("fecha_publicacion_dt", ascending=False, inplace=True)

    combined.reset_index(drop=True, inplace=True)

    logger.info(
        "DataFrame final: %d artículos, %d columnas, fuentes: %s",
        len(combined),
        len(combined.columns),
        combined["source_file"].unique().tolist() if "source_file" in combined.columns else [],
    )

    return combined


# Uso en notebook:
# df = load_articles_df("data/parsed_json")
# df.head()


In [2]:
# Ejecuta la carga directa de todos los JSON en data/parsed_json a un DataFrame
import os

# Notebook path: a:/programing_workspace/SYJ_webScrapper/src/dataframe/
_notebook_dir = Path(os.path.abspath(""))   # cwd when running the notebook
_repo_root = _notebook_dir

# Walk up until we find the repo root (contains "data" folder)
for _ in range(5):
    if (_repo_root / "data" / "parsed_json").is_dir():
        break
    _repo_root = _repo_root.parent

candidates = [
    _repo_root / "data" / "parsed_json",
    Path("data") / "parsed_json",
    Path(script_path).parents[2] / "data" / "parsed_json" if "script_path" in globals() else None,
    Path(path).parents[2] / "data" / "parsed_json" if "path" in globals() else None,
]

parsed_dir = next((p.resolve() for p in candidates if p and p.is_dir()), None)
if parsed_dir is None:
    raise FileNotFoundError(f"Directorio no encontrado en rutas probadas: {[str(p) for p in candidates if p]}")

df = load_articles_df(parsed_dir)

print(f"Artículos cargados: {len(df)}")
display(df.head())

Artículos cargados: 105


,fecha_obtencion,fecha_publicacion,fuente,raw_text,snippet_text,url_noticia,title,fetcher,id_noticia,source_file,fecha_publicacion_dt,fecha_obtencion_dt
0,12/03/2026,12/03/2026,Qhubo Cali,MUNDO CURIOSO,MUNDO CURIOSO,https://www.qhubocali.com/categoria/mundo-curioso,Capturado un colombiano por lanzarle una roca ...,cloudflare,35,qhubo_cali,2026-03-12,2026-03-12
1,12/03/2026,12/03/2026,El Pais Cali,La Unidad Administrativa Especial de Protecció...,La Unidad Administrativa Especial de Protecció...,https://www.elpais.com.co/cali/esterilizacion-...,Esterilización gratis en Cali para perros y ga...,requests,27,el_pais_cali,2026-03-12,2026-03-12
2,12/03/2026,12/03/2026,El Pais Cali,Sigue la disputa en el Valle del Cauca por la ...,Sigue la disputa en el Valle del Cauca por la ...,https://www.elpais.com.co/cali/incertidumbre-p...,Incertidumbre por la última curul a la Cámara ...,requests,17,el_pais_cali,2026-03-12,2026-03-12
3,12/03/2026,12/03/2026,El Pais Cali,"Un trabajo conjunto entre la Policía Nacional,...","Un trabajo conjunto entre la Policía Nacional,...",https://www.elpais.com.co/judicial/policia-nac...,Policía Nacional incauta histórico cargamento ...,cloudflare,19,el_pais_cali,2026-03-12,2026-03-12
4,12/03/2026,12/03/2026,El Pais Cali,Un nuevo ataque con explosivos se registró en ...,Un nuevo ataque con explosivos se registró en ...,https://www.elpais.com.co/judicial/cinco-polic...,Cinco policías heridos tras ataque con explosi...,requests,20,el_pais_cali,2026-03-12,2026-03-12


In [ ]:
import sys
import subprocess

# Dependencias instalables para ese bloque de imports:
# - numpy
# - IPython
# - plotly
# (re, unicodedata y collections son parte de la librería estándar de Python)


subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "numpy", "ipython", "plotly", "torch"])

print("Instalación completada.")

In [16]:
import re
import unicodedata
from collections import Counter
import numpy as np

# -----------------------------------
# 3.0) Restauración previa al bloque 3.5 (sin borrar lo nuevo)
# -----------------------------------


def normalize_text(text: str) -> str:
    text = "" if pd.isna(text) else str(text)
    text = text.lower().strip()
    text = "".join(
        c for c in unicodedata.normalize("NFD", text)
        if unicodedata.category(c) != "Mn"
    )
    text = re.sub(r"[^a-z0-9áéíóúüñ\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Base para análisis (prioriza ventana reciente ya preparada)
if "window_30d" in globals() and isinstance(window_30d, pd.DataFrame) and not window_30d.empty:
    df_analysis = window_30d.copy()
elif "df" in globals() and isinstance(df, pd.DataFrame) and not df.empty:
    df_analysis = df.copy()
else:
    df_analysis = pd.DataFrame()

if df_analysis.empty:
    print("No hay datos para restaurar el pipeline previo al 3.5.")
else:
    # -----------------------------------
    # 3.1) Normalización base
    # -----------------------------------
    if "fecha_publicacion_dt" not in df_analysis.columns and "fecha_publicacion" in df_analysis.columns:
        df_analysis["fecha_publicacion_dt"] = pd.to_datetime(
            df_analysis["fecha_publicacion"], dayfirst=True, errors="coerce"
        )

    if "texto" not in df_analysis.columns:
        if "raw_text" in df_analysis.columns:
            df_analysis["texto"] = df_analysis["raw_text"].fillna("")
        elif "snippet_text" in df_analysis.columns:
            df_analysis["texto"] = df_analysis["snippet_text"].fillna("")
        elif "title" in df_analysis.columns:
            df_analysis["texto"] = df_analysis["title"].fillna("")
        else:
            df_analysis["texto"] = ""

    if "texto_norm" not in df_analysis.columns:
        df_analysis["texto_norm"] = df_analysis["texto"].apply(normalize_text)

    # -----------------------------------
    # 3.2) Sentimiento (fallback si faltaba)
    # -----------------------------------
    if "sentiment_score" not in df_analysis.columns:
        neg_words = {"homicidio", "asesinato", "ataque", "explosivo", "muerto", "herido", "terror", "secuestro", "extorsion", "robo"}
        pos_words = {"captura", "incautacion", "prevencion", "jornada", "comunidad", "control", "seguridad"}

        def score_sent(txt: str) -> float:
            t = normalize_text(txt)
            n_neg = sum(1 for w in neg_words if w in t)
            n_pos = sum(1 for w in pos_words if w in t)
            den = max(1, n_neg + n_pos)
            return (n_pos - n_neg) / den

        df_analysis["sentiment_score"] = df_analysis["texto_norm"].apply(score_sent).astype(float)

    if "sentiment_label" not in df_analysis.columns:
        df_analysis["sentiment_label"] = np.select(
            [df_analysis["sentiment_score"] < -0.05, df_analysis["sentiment_score"] > 0.05],
            ["Negativo", "Positivo"],
            default="Neutro",
        )

    # Probabilidades proxy para compatibilidad
    if "sent_neg" not in df_analysis.columns:
        df_analysis["sent_neg"] = (df_analysis["sentiment_score"].clip(upper=0).abs()).clip(0, 1)
    if "sent_pos" not in df_analysis.columns:
        df_analysis["sent_pos"] = (df_analysis["sentiment_score"].clip(lower=0)).clip(0, 1)
    if "sent_neu" not in df_analysis.columns:
        df_analysis["sent_neu"] = (1 - (df_analysis["sent_neg"] + df_analysis["sent_pos"])).clip(0, 1)

    # -----------------------------------
    # 3.3) Clasificación de hecho (reglas base)
    # -----------------------------------
    EVENT_RULES_BASE = {
        "Explosivos/Terror": [r"\bexplosiv", r"\bbomba", r"\batentad", r"\bhostigamiento", r"\bterror"],
        "Violencia letal": [r"\bhomicid", r"\basesinad", r"\bmuert[oa]s?\b", r"\bsicari"],
        "Secuestro/Extorsión": [r"\bsecuestro", r"\bextors", r"\bvacuna\b", r"\bplagi[oa]"],
        "Crimen organizado": [r"\bnarcotraf", r"\bdisidenc", r"\bgrupo[s]? armad", r"\bmaf"],
        "Delitos patrimoniales": [r"\bhurto", r"\brobo", r"\bestafa", r"\batraco", r"\bfleteo"],
        "Accidentes y emergencias": [r"\baccident", r"\bemergenc", r"\bincendio", r"\bchoque", r"\bderrumbe"],
        "Judicial/Control": [r"\bcaptur", r"\bimput", r"\bfiscal", r"\bpolicia", r"\boperativo", r"\bincauta"],
        "Convivencia social": [r"\bri[aá]s?\b", r"\bintolerancia", r"\bcomunidad", r"\bvecin", r"\bconvivencia"],
        "Prevención/Comunidad": [r"\bprevenci", r"\bcampana", r"\bjornada", r"\bsecretaria", r"\bprograma"],
    }

    def classify_event(txt: str):
        t = normalize_text(txt)
        hits = {cat: sum(1 for p in pats if re.search(p, t)) for cat, pats in EVENT_RULES_BASE.items()}
        best = max(hits, key=hits.get)
        n = hits[best]
        if n <= 0:
            return "Sin clasificar", 0.0
        return best, min(0.95, 0.45 + 0.10 * n)

    if "hecho_categoria" not in df_analysis.columns or "hecho_confianza" not in df_analysis.columns:
        tmp = df_analysis["texto_norm"].apply(classify_event).apply(pd.Series)
        tmp.columns = ["hecho_categoria_tmp", "hecho_confianza_tmp"]
        if "hecho_categoria" not in df_analysis.columns:
            df_analysis["hecho_categoria"] = tmp["hecho_categoria_tmp"]
        if "hecho_confianza" not in df_analysis.columns:
            df_analysis["hecho_confianza"] = tmp["hecho_confianza_tmp"]

    # -----------------------------------
    # 3.4) Severidad, riesgo y salidas de soporte (tops/trends)
    # -----------------------------------
    if "severity_map" not in globals():
        severity_map = {
            "Violencia letal": 5, "Explosivos/Terror": 5, "Secuestro/Extorsión": 4,
            "Crimen organizado": 4, "Delitos patrimoniales": 3, "Accidentes y emergencias": 3,
            "Convivencia social": 2, "Judicial/Control": 2, "Prevención/Comunidad": 1, "Sin clasificar": 1
        }

    critical_terms = [r"\bmuert", r"\bherid", r"\bexplosiv", r"\bsecuestro", r"\bextors", r"\batentad"]
    if "critical_hits" not in df_analysis.columns:
        df_analysis["critical_hits"] = df_analysis["texto_norm"].apply(
            lambda x: int(sum(1 for p in critical_terms if re.search(p, x)))
        )

    if "severity" not in df_analysis.columns:
        df_analysis["severity"] = df_analysis["hecho_categoria"].map(severity_map).fillna(1).astype(int)

    if "risk_score" not in df_analysis.columns:
        df_analysis["risk_score"] = (
            (df_analysis["severity"] * 12)
            + ((-pd.to_numeric(df_analysis["sentiment_score"], errors="coerce").fillna(0)).clip(lower=0) * 30)
            + (df_analysis["critical_hits"] * 10)
        ).clip(0, 100).round(1)

    # Resúmenes usados por la celda de reporte
    total_noticias = int(len(df_analysis))

    trend = (
        df_analysis.dropna(subset=["fecha_publicacion_dt"])
        .groupby([pd.Grouper(key="fecha_publicacion_dt", freq="W"), "hecho_categoria"], dropna=False)
        .size()
        .reset_index(name="conteo")
    )

    trend_daily = (
        df_analysis.dropna(subset=["fecha_publicacion_dt"])
        .groupby(df_analysis["fecha_publicacion_dt"].dt.date)
        .agg(noticias=("title", "count") if "title" in df_analysis.columns else ("texto", "count"),
             riesgo_medio=("risk_score", "mean"))
        .reset_index()
        .rename(columns={"fecha_publicacion_dt": "fecha"})
    )

    source_cat = (
        df_analysis.groupby(["fuente", "hecho_categoria"], dropna=False)
        .size().reset_index(name="conteo")
        if {"fuente", "hecho_categoria"}.issubset(df_analysis.columns)
        else pd.DataFrame(columns=["fuente", "hecho_categoria", "conteo"])
    )
    source_cat_html = source_cat.to_html(index=False)

    cols_top = [c for c in ["fecha_publicacion_dt", "fuente", "title", "hecho_categoria", "hecho_confianza", "risk_score", "url_noticia"] if c in df_analysis.columns]
    top_risk = df_analysis.sort_values("risk_score", ascending=False)[cols_top].head(10).copy()
    top_risk_view = top_risk.copy()
    if "url_noticia" in top_risk_view.columns:
        top_risk_view["url_noticia"] = top_risk_view["url_noticia"].apply(
            lambda u: f'<a href="{u}" target="_blank">ver</a>' if pd.notna(u) else ""
        )
    top_risk_html = top_risk_view.to_html(index=False, escape=False)

    if "stopwords_es" not in globals():
        stopwords_es = {"de", "la", "el", "en", "y", "a", "los", "las", "del", "que", "por", "con", "para", "un", "una"}

    tokens = []
    for tx in df_analysis["texto_norm"].fillna(""):
        tokens.extend([w for w in tx.split() if len(w) > 3 and w not in stopwords_es and not w.isdigit()])

    top_terms = pd.DataFrame(Counter(tokens).most_common(20), columns=["termino", "frecuencia"])
    top_terms_view = top_terms.copy()
    top_terms_html = top_terms_view.to_html(index=False)

    print(f"Restauración previa al 3.5 completada. Registros: {len(df_analysis)}")# -----------------------------------
# 3.5) Adaptación de modelos al dominio + filtro geográfico (Cali) + métricas
# -----------------------------------
try:
    from sklearn.pipeline import Pipeline
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])
    from sklearn.pipeline import Pipeline
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# Contenedores de métricas y matrices
model_metrics = {"sentiment": {}, "event": {}, "prediction": {}}
sent_cm_train_df = pd.DataFrame()
sent_cm_val_df = pd.DataFrame()
evt_cm_train_df = pd.DataFrame()
evt_cm_val_df = pd.DataFrame()
sent_report_train_df = pd.DataFrame()
sent_report_val_df = pd.DataFrame()
evt_report_train_df = pd.DataFrame()
evt_report_val_df = pd.DataFrame()

# --- A) Filtro geográfico: descartar noticias fuera de Santiago de Cali ---
CALI_PATTERNS = [
    r"\bsantiago de cali\b", r"\bcali\b", r"\bvalle del cauca\b", r"\byumbo\b", r"\bjamundi\b", r"\bpalmira\b"
]

OTHER_CITY_PATTERNS = {
    "bogota": r"\bbogota\b",
    "medellin": r"\bmedellin\b",
    "barranquilla": r"\bbarranquilla\b",
    "cartagena": r"\bcartagena\b",
    "bucaramanga": r"\bbucaramanga\b",
    "cucuta": r"\bcucuta\b",
    "pasto": r"\bpasto\b",
    "pereira": r"\bpereira\b",
    "manizales": r"\bmanizales\b",
    "ibague": r"\bibague\b",
    "armenia": r"\barmenia\b",
    "neiva": r"\bneiva\b",
    "villavicencio": r"\bvillavicencio\b",
    "santa marta": r"\bsanta marta\b",
}

def geo_flags(text: str):
    t = normalize_text(text)
    in_cali = any(re.search(p, t) for p in CALI_PATTERNS)
    other_hits = [k for k, p in OTHER_CITY_PATTERNS.items() if re.search(p, t)]
    outside_cali = (not in_cali) and (len(other_hits) > 0)
    return in_cali, outside_cali, ", ".join(other_hits[:3])

geo = df_analysis["texto"].fillna("").apply(geo_flags)
df_analysis["menciona_cali"] = geo.apply(lambda x: x[0])
df_analysis["outside_cali"] = geo.apply(lambda x: x[1])
df_analysis["outside_places"] = geo.apply(lambda x: x[2])

df_fuera_cali = df_analysis[df_analysis["outside_cali"]].copy()
df_perception = df_analysis[~df_analysis["outside_cali"]].copy()

print(f"Noticias totales: {len(df_analysis)}")
print(f"Descartadas por fuera de Cali: {len(df_fuera_cali)}")
print(f"Usadas para percepción de seguridad en Cali: {len(df_perception)}")

# --- B) Fine-tuning ligero de sentimiento + métricas train/val ---
use_ft_sent = False
sent_pipe_ft = None

sent_train = df_perception[
    df_perception["sentiment_label"].isin(["Negativo", "Neutro", "Positivo"]) &
    df_perception["texto_norm"].fillna("").ne("")
].copy()

sent_counts = sent_train["sentiment_label"].value_counts()

if len(sent_counts) >= 2 and sent_counts.min() >= 8:
    use_ft_sent = True

    Xs = sent_train["texto_norm"]
    ys = sent_train["sentiment_label"]

    try:
        Xs_tr, Xs_val, ys_tr, ys_val = train_test_split(
            Xs, ys, test_size=0.2, random_state=42, stratify=ys
        )
    except Exception:
        Xs_tr, Xs_val, ys_tr, ys_val = Xs, Xs, ys, ys

    sent_pipe_ft = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=40000, min_df=2)),
        ("clf", LogisticRegression(max_iter=2500, class_weight="balanced"))
    ])

    sent_pipe_ft.fit(Xs_tr, ys_tr)

    ys_tr_pred = sent_pipe_ft.predict(Xs_tr)
    ys_val_pred = sent_pipe_ft.predict(Xs_val)
    sent_labels_order = sorted(ys.unique().tolist())

    sent_cm_train = confusion_matrix(ys_tr, ys_tr_pred, labels=sent_labels_order)
    sent_cm_val = confusion_matrix(ys_val, ys_val_pred, labels=sent_labels_order)
    sent_cm_train_df = pd.DataFrame(sent_cm_train, index=sent_labels_order, columns=sent_labels_order)
    sent_cm_val_df = pd.DataFrame(sent_cm_val, index=sent_labels_order, columns=sent_labels_order)

    sent_rep_tr = classification_report(ys_tr, ys_tr_pred, output_dict=True, zero_division=0)
    sent_rep_val = classification_report(ys_val, ys_val_pred, output_dict=True, zero_division=0)
    sent_report_train_df = pd.DataFrame(sent_rep_tr).T
    sent_report_val_df = pd.DataFrame(sent_rep_val).T

    model_metrics["sentiment"] = {
        "train_accuracy": float(accuracy_score(ys_tr, ys_tr_pred)),
        "val_accuracy": float(accuracy_score(ys_val, ys_val_pred)),
        "train_f1_macro": float(f1_score(ys_tr, ys_tr_pred, average="macro")),
        "val_f1_macro": float(f1_score(ys_val, ys_val_pred, average="macro")),
        "n_train": int(len(Xs_tr)),
        "n_val": int(len(Xs_val)),
        "class_counts": sent_counts.to_dict(),
    }

    # Reentrenar con todo el set disponible para inferencia final
    sent_pipe_ft.fit(Xs, ys)

    sent_pred = sent_pipe_ft.predict(df_perception["texto_norm"])
    sent_proba = sent_pipe_ft.predict_proba(df_perception["texto_norm"])
    sent_classes = list(sent_pipe_ft.named_steps["clf"].classes_)

    idx_neg = sent_classes.index("Negativo") if "Negativo" in sent_classes else None
    idx_pos = sent_classes.index("Positivo") if "Positivo" in sent_classes else None

    if idx_neg is not None and idx_pos is not None:
        sent_score_ft = sent_proba[:, idx_pos] - sent_proba[:, idx_neg]
    else:
        sent_score_ft = np.zeros(len(df_perception))

    df_perception["sentiment_label_ft"] = sent_pred
    df_perception["sentiment_score_ft"] = sent_score_ft
else:
    df_perception["sentiment_label_ft"] = df_perception["sentiment_label"]
    df_perception["sentiment_score_ft"] = df_perception["sentiment_score"]
    model_metrics["sentiment"] = {"enabled": False, "reason": "Muestras insuficientes por clase."}

# --- C) Fine-tuning ligero de clasificación de hechos (mejorado) ---
# Reglas débiles para reducir "Sin clasificar"
EVENT_RULES = {
    "Explosivos/Terror": [r"\bexplosiv", r"\bbomba", r"\batentad", r"\bcarro bomba", r"\bhostigamiento", r"\bterror"],
    "Violencia letal": [r"\bhomicid", r"\basesinad", r"\bmuert[oa]s?\b", r"\bfallecid", r"\bsicari"],
    "Secuestro/Extorsión": [r"\bsecuestro", r"\bextors", r"\bvacuna\b", r"\bplagi[oa]"],
    "Crimen organizado": [r"\bnarcotraf", r"\bdisidenc", r"\bclan\b", r"\bgrupo[s]? armad", r"\bmaf"],
    "Delitos patrimoniales": [r"\bhurto", r"\brobo", r"\bfleteo", r"\bestafa", r"\batraco"],
    "Accidentes y emergencias": [r"\baccident", r"\bemergenc", r"\bincendio", r"\bderrumbe", r"\bchoque"],
    "Judicial/Control": [r"\bcaptur", r"\bimput", r"\bfiscal", r"\bpolicia", r"\boperativo", r"\bincauta"],
    "Convivencia social": [r"\bri[aá]s?\b", r"\bintolerancia", r"\bcomunidad", r"\bvecin", r"\bconvivencia"],
    "Prevención/Comunidad": [r"\bprevenci", r"\bcampana", r"\bjornada", r"\bsecretaria", r"\bprograma", r"\bestiriliza|esteriliza"],
}

def weak_event_label(text: str):
    tx = normalize_text(text)
    scores = {}
    for cat, pats in EVENT_RULES.items():
        scores[cat] = sum(1 for p in pats if re.search(p, tx))
    best_cat = max(scores, key=scores.get)
    best_hits = scores[best_cat]
    if best_hits <= 0:
        return "Sin clasificar", 0
    return best_cat, int(best_hits)

df_perception[["hecho_categoria_rule", "rule_hits"]] = (
    df_perception["texto_norm"].fillna("").apply(weak_event_label).apply(pd.Series)
)

use_ft_evt = False
evt_pipe_ft = None

# Seeds confiables
evt_seed = df_perception[
    (df_perception["hecho_categoria"] != "Sin clasificar") &
    (df_perception["hecho_confianza"] >= 0.55) &
    df_perception["texto_norm"].fillna("").ne("")
][["texto_norm", "hecho_categoria"]].copy()
evt_seed["w"] = 1.0

# Weak labels para rescatar "Sin clasificar" o baja confianza
evt_weak = df_perception[
    ((df_perception["hecho_categoria"] == "Sin clasificar") | (df_perception["hecho_confianza"] < 0.55)) &
    (df_perception["hecho_categoria_rule"] != "Sin clasificar") &
    (df_perception["rule_hits"] >= 1)
][["texto_norm", "hecho_categoria_rule"]].copy()
evt_weak = evt_weak.rename(columns={"hecho_categoria_rule": "hecho_categoria"})
evt_weak["w"] = 0.55

evt_train = pd.concat([evt_seed, evt_weak], ignore_index=True)
evt_counts = evt_train["hecho_categoria"].value_counts()

if len(evt_counts) >= 3 and evt_counts.min() >= 3:
    use_ft_evt = True

    Xe = evt_train["texto_norm"]
    ye = evt_train["hecho_categoria"]
    we = evt_train["w"].values

    try:
        Xe_tr, Xe_val, ye_tr, ye_val, we_tr, we_val = train_test_split(
            Xe, ye, we, test_size=0.2, random_state=42, stratify=ye
        )
    except Exception:
        Xe_tr, Xe_val, ye_tr, ye_val, we_tr, we_val = Xe, Xe, ye, ye, we, we

    evt_pipe_ft = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=50000, min_df=2)),
        ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", multi_class="auto"))
    ])

    evt_pipe_ft.fit(Xe_tr, ye_tr, clf__sample_weight=we_tr)

    ye_tr_pred = evt_pipe_ft.predict(Xe_tr)
    ye_val_pred = evt_pipe_ft.predict(Xe_val)

    evt_labels_order = sorted(ye.unique().tolist())
    evt_cm_train = confusion_matrix(ye_tr, ye_tr_pred, labels=evt_labels_order)
    evt_cm_val = confusion_matrix(ye_val, ye_val_pred, labels=evt_labels_order)
    evt_cm_train_df = pd.DataFrame(evt_cm_train, index=evt_labels_order, columns=evt_labels_order)
    evt_cm_val_df = pd.DataFrame(evt_cm_val, index=evt_labels_order, columns=evt_labels_order)

    evt_rep_tr = classification_report(ye_tr, ye_tr_pred, output_dict=True, zero_division=0)
    evt_rep_val = classification_report(ye_val, ye_val_pred, output_dict=True, zero_division=0)
    evt_report_train_df = pd.DataFrame(evt_rep_tr).T
    evt_report_val_df = pd.DataFrame(evt_rep_val).T

    model_metrics["event"] = {
        "train_accuracy": float(accuracy_score(ye_tr, ye_tr_pred)),
        "val_accuracy": float(accuracy_score(ye_val, ye_val_pred)),
        "train_f1_macro": float(f1_score(ye_tr, ye_tr_pred, average="macro")),
        "val_f1_macro": float(f1_score(ye_val, ye_val_pred, average="macro")),
        "n_train": int(len(Xe_tr)),
        "n_val": int(len(Xe_val)),
        "class_counts": evt_counts.to_dict(),
        "n_categories": int(evt_train["hecho_categoria"].nunique()),
    }

    # Reentrenar con todo
    evt_pipe_ft.fit(Xe, ye, clf__sample_weight=we)

    evt_pred = evt_pipe_ft.predict(df_perception["texto_norm"])
    evt_proba_all = evt_pipe_ft.predict_proba(df_perception["texto_norm"])
    evt_conf = evt_proba_all.max(axis=1)

    df_perception["hecho_categoria_ft"] = evt_pred
    df_perception["hecho_confianza_ft"] = evt_conf

    # Fallback rule-based para confianzas muy bajas
    low_conf_mask = df_perception["hecho_confianza_ft"] < 0.45
    rule_ok_mask = df_perception["hecho_categoria_rule"] != "Sin clasificar"
    apply_rule_mask = low_conf_mask & rule_ok_mask
    df_perception.loc[apply_rule_mask, "hecho_categoria_ft"] = df_perception.loc[apply_rule_mask, "hecho_categoria_rule"]
    df_perception.loc[apply_rule_mask, "hecho_confianza_ft"] = 0.50
else:
    df_perception["hecho_categoria_ft"] = df_perception["hecho_categoria"]
    df_perception["hecho_confianza_ft"] = df_perception["hecho_confianza"]
    model_metrics["event"] = {"enabled": False, "reason": "Muestras/categorías insuficientes para FT de hechos."}

# --- D) Ensamble final (reglas + FT) ---
df_perception["sentiment_label_final"] = df_perception["sentiment_label"]
df_perception["sentiment_score_final"] = df_perception["sentiment_score"]

mask_sent = (df_perception["sentiment_label"] == "Neutro") | (df_perception["sentiment_score"].abs() < 0.20)
df_perception.loc[mask_sent, "sentiment_label_final"] = df_perception.loc[mask_sent, "sentiment_label_ft"]
df_perception.loc[mask_sent, "sentiment_score_final"] = df_perception.loc[mask_sent, "sentiment_score_ft"]

df_perception["hecho_categoria_final"] = df_perception["hecho_categoria"]
df_perception["hecho_confianza_final"] = df_perception["hecho_confianza"]

mask_evt = (
    (df_perception["hecho_categoria"] == "Sin clasificar") |
    (df_perception["hecho_confianza"] < 0.65)
)
df_perception.loc[mask_evt, "hecho_categoria_final"] = df_perception.loc[mask_evt, "hecho_categoria_ft"]
df_perception.loc[mask_evt, "hecho_confianza_final"] = df_perception.loc[mask_evt, "hecho_confianza_ft"]

# Último rescate de "Sin clasificar" usando reglas
mask_still_unclass = (df_perception["hecho_categoria_final"] == "Sin clasificar") & (df_perception["hecho_categoria_rule"] != "Sin clasificar")
df_perception.loc[mask_still_unclass, "hecho_categoria_final"] = df_perception.loc[mask_still_unclass, "hecho_categoria_rule"]
df_perception.loc[mask_still_unclass, "hecho_confianza_final"] = np.maximum(
    df_perception.loc[mask_still_unclass, "hecho_confianza_final"].values, 0.45
)

# Recalcular severidad/riesgo con columnas finales
df_perception["severity"] = df_perception["hecho_categoria_final"].map(severity_map).fillna(1).astype(int)
df_perception["risk_score"] = (
    (df_perception["severity"] * 12)
    + ((-df_perception["sentiment_score_final"]).clip(lower=0) * 30)
    + (df_perception["critical_hits"] * 10)
).clip(0, 100).round(1)

# Métricas de predicción final
sin_before = int((df_perception["hecho_categoria"] == "Sin clasificar").sum())
sin_after = int((df_perception["hecho_categoria_final"] == "Sin clasificar").sum())
total_pred = int(len(df_perception))
sin_before_pct = (sin_before / total_pred * 100) if total_pred else 0.0
sin_after_pct = (sin_after / total_pred * 100) if total_pred else 0.0

model_metrics["prediction"] = {
    "n_total": total_pred,
    "sin_clasificar_before": sin_before,
    "sin_clasificar_after": sin_after,
    "sin_before_pct": float(sin_before_pct),
    "sin_after_pct": float(sin_after_pct),
    "n_categorias_finales": int(df_perception["hecho_categoria_final"].nunique()),
}

# Reemplazo para el resto del pipeline (alertas/gráficas/KPIs)
df_analysis = df_perception.copy()

# Guardar métricas y matrices en attrs de df_analysis
df_analysis.attrs["model_metrics"] = model_metrics
df_analysis.attrs["sent_cm_train_df"] = sent_cm_train_df
df_analysis.attrs["sent_cm_val_df"] = sent_cm_val_df
df_analysis.attrs["evt_cm_train_df"] = evt_cm_train_df
df_analysis.attrs["evt_cm_val_df"] = evt_cm_val_df
df_analysis.attrs["sent_report_train_df"] = sent_report_train_df
df_analysis.attrs["sent_report_val_df"] = sent_report_val_df
df_analysis.attrs["evt_report_train_df"] = evt_report_train_df
df_analysis.attrs["evt_report_val_df"] = evt_report_val_df

# También en columnas-resumen para trazabilidad en df_analysis
df_analysis["metric_sin_before_pct"] = sin_before_pct
df_analysis["metric_sin_after_pct"] = sin_after_pct
df_analysis["metric_event_categories_final"] = model_metrics["prediction"]["n_categorias_finales"]
df_analysis["metric_use_ft_sent"] = bool(use_ft_sent)
df_analysis["metric_use_ft_evt"] = bool(use_ft_evt)

print("\n=== Resumen métricas ===")
print(pd.DataFrame(model_metrics).T)
print("\nSin clasificar antes:", sin_before, f"({sin_before_pct:.2f}%)")
print("Sin clasificar después:", sin_after, f"({sin_after_pct:.2f}%)")
print("Categorías finales detectadas:", model_metrics["prediction"]["n_categorias_finales"])


Restauración previa al 3.5 completada. Registros: 98
Noticias totales: 98
Descartadas por fuera de Cali: 11
Usadas para percepción de seguridad en Cali: 87

=== Resumen métricas ===
           train_accuracy val_accuracy train_f1_macro val_f1_macro n_train  \
sentiment        0.985507     0.722222       0.988769     0.525926      69   
event                 NaN          NaN            NaN          NaN     NaN   
prediction            NaN          NaN            NaN          NaN     NaN   

           n_val                                    class_counts enabled  \
sentiment     18  {'Neutro': 44, 'Positivo': 33, 'Negativo': 10}     NaN   
event        NaN                                             NaN   False   
prediction   NaN                                             NaN     NaN   

                                                       reason n_total  \
sentiment                                                 NaN     NaN   
event       Muestras/categorías insuficientes para FT 

In [23]:
from pathlib import Path

# Reporte HTML: Percepción de seguridad en Cali (versión gráfica y precisa)

import plotly.express as px
import plotly.graph_objects as go

# -----------------------------
# 1) Base de datos para reporte
# -----------------------------
if "df_analysis" in globals() and isinstance(df_analysis, pd.DataFrame) and not df_analysis.empty:
    report_df = df_analysis.copy()
elif "window_30d" in globals() and isinstance(window_30d, pd.DataFrame) and not window_30d.empty:
    report_df = window_30d.copy()
else:
    raise ValueError("No hay datos disponibles en df_analysis/window_30d para construir el reporte.")

# Columnas finales (fallback a base)
cat_col = "hecho_categoria_final" if "hecho_categoria_final" in report_df.columns else "hecho_categoria"
sent_col = "sentiment_label_final" if "sentiment_label_final" in report_df.columns else "sentiment_label"
score_col = "sentiment_score_final" if "sentiment_score_final" in report_df.columns else "sentiment_score"
conf_col = "hecho_confianza_final" if "hecho_confianza_final" in report_df.columns else "hecho_confianza"

if "fecha_publicacion_dt" not in report_df.columns and "fecha_publicacion" in report_df.columns:
    report_df["fecha_publicacion_dt"] = pd.to_datetime(report_df["fecha_publicacion"], dayfirst=True, errors="coerce")

report_df["fecha"] = pd.to_datetime(report_df["fecha_publicacion_dt"], errors="coerce").dt.date

# -----------------------------
# 2) KPIs ejecutivos
# -----------------------------
total = int(len(report_df))
negativos = int((report_df[sent_col] == "Negativo").sum()) if sent_col in report_df.columns else 0
neutros = int((report_df[sent_col] == "Neutro").sum()) if sent_col in report_df.columns else 0
positivos = int((report_df[sent_col] == "Positivo").sum()) if sent_col in report_df.columns else 0

pct_neg = (negativos / total * 100) if total else 0.0
pct_neu = (neutros / total * 100) if total else 0.0
pct_pos = (positivos / total * 100) if total else 0.0

alto_riesgo = int((report_df["risk_score"] >= 70).sum()) if "risk_score" in report_df.columns else 0
pct_alto_riesgo = (alto_riesgo / total * 100) if total else 0.0

sin_cls = int((report_df[cat_col] == "Sin clasificar").sum()) if cat_col in report_df.columns else 0
pct_sin_cls = (sin_cls / total * 100) if total else 0.0

# -----------------------------
# 3) Gráficas
# -----------------------------
# 3.1 Tendencia diaria (volumen + riesgo)
daily = (
    report_df.dropna(subset=["fecha"])
    .groupby("fecha", as_index=False)
    .agg(
        noticias=("title", "count") if "title" in report_df.columns else ("texto", "count"),
        riesgo_medio=("risk_score", "mean") if "risk_score" in report_df.columns else ("fecha", "count"),
    )
)

fig_daily = go.Figure()
fig_daily.add_trace(go.Bar(x=daily["fecha"], y=daily["noticias"], name="Noticias", yaxis="y1"))
if "risk_score" in report_df.columns:
    fig_daily.add_trace(go.Scatter(x=daily["fecha"], y=daily["riesgo_medio"], name="Riesgo medio", mode="lines+markers", yaxis="y2"))

fig_daily.update_layout(
    title="Tendencia diaria de cobertura y riesgo",
    template="plotly_white",
    xaxis_title="Fecha",
    yaxis=dict(title="Noticias"),
    yaxis2=dict(title="Riesgo medio", overlaying="y", side="right"),
    legend=dict(orientation="h", y=1.12),
)

# 3.2 Distribución por categoría de hecho
cat_dist = (
    report_df.groupby(cat_col, dropna=False)
    .size()
    .reset_index(name="conteo")
    .sort_values("conteo", ascending=False)
)

fig_cat = px.bar(
    cat_dist,
    x="conteo",
    y=cat_col,
    orientation="h",
    title="Distribución de hechos de seguridad (clasificación final)",
    template="plotly_white",
)

# 3.3 Matriz fuente vs categoría
if "fuente" in report_df.columns:
    src_cat_final = report_df.groupby(["fuente", cat_col], dropna=False).size().reset_index(name="conteo")
    heat = src_cat_final.pivot(index="fuente", columns=cat_col, values="conteo").fillna(0)
    fig_heat = px.imshow(
        heat,
        text_auto=True,
        aspect="auto",
        color_continuous_scale="Blues",
        title="Intensidad de cobertura por fuente y categoría",
    )
else:
    src_cat_final = pd.DataFrame(columns=["fuente", cat_col, "conteo"])
    fig_heat = go.Figure().update_layout(title="Sin columna 'fuente' disponible", template="plotly_white")

# 3.4 Sentimiento por fuente
if {"fuente", sent_col}.issubset(report_df.columns):
    sent_src = report_df.groupby(["fuente", sent_col]).size().reset_index(name="conteo")
    fig_sent_src = px.bar(
        sent_src,
        x="fuente",
        y="conteo",
        color=sent_col,
        barmode="stack",
        title="Composición de sentimiento por fuente",
        template="plotly_white",
        category_orders={sent_col: ["Negativo", "Neutro", "Positivo"]},
    )
else:
    fig_sent_src = go.Figure().update_layout(title="Sin columnas para sentimiento por fuente", template="plotly_white")

# -----------------------------
# 4) Tablas clave
# -----------------------------
# Top riesgo
cols_top = [c for c in ["fecha_publicacion_dt", "fuente", "title", cat_col, conf_col, "risk_score", "url_noticia"] if c in report_df.columns]
top10 = report_df.sort_values("risk_score", ascending=False)[cols_top].head(10).copy() if "risk_score" in report_df.columns else pd.DataFrame()

if "url_noticia" in top10.columns:
    top10["url_noticia"] = top10["url_noticia"].apply(lambda u: f'<a href="{u}" target="_blank">ver</a>' if pd.notna(u) else "")

top10_html = top10.to_html(index=False, escape=False) if not top10.empty else "<p>No disponible.</p>"

# Top términos (reusar si existe)
if "top_terms" in globals() and isinstance(top_terms, pd.DataFrame) and not top_terms.empty:
    terms_html = top_terms.head(20).to_html(index=False)
else:
    toks = []
    sw = stopwords_es if "stopwords_es" in globals() else set()
    for tx_ in report_df["texto_norm"].fillna("") if "texto_norm" in report_df.columns else []:
        toks.extend([w for w in str(tx_).split() if len(w) > 3 and w not in sw and not w.isdigit()])
    tt = pd.DataFrame(Counter(toks).most_common(20), columns=["termino", "frecuencia"]) if len(toks) else pd.DataFrame(columns=["termino", "frecuencia"])
    terms_html = tt.to_html(index=False)

# Resumen FT/modelo
metrics = report_df.attrs.get("model_metrics", {}) if hasattr(report_df, "attrs") else {}
metrics_html = pd.DataFrame(metrics).T.to_html() if metrics else "<p>No hay métricas de modelo en attrs.</p>"

# -----------------------------
# 5) Render HTML final
# -----------------------------
plot_daily_html = fig_daily.to_html(full_html=False, include_plotlyjs="cdn")
plot_cat_html = fig_cat.to_html(full_html=False, include_plotlyjs=False)
plot_heat_html = fig_heat.to_html(full_html=False, include_plotlyjs=False)
plot_sent_src_html = fig_sent_src.to_html(full_html=False, include_plotlyjs=False)

insight_1 = f"El {pct_neg:.1f}% de la muestra tiene tono negativo; esto sugiere percepción de riesgo sostenida."
insight_2 = f"Las notas de alto riesgo (>=70) representan {pct_alto_riesgo:.1f}% del total ({alto_riesgo}/{total})."
insight_3 = f"La tasa de 'Sin clasificar' cerró en {pct_sin_cls:.1f}% ({sin_cls}/{total})."

if "sin_before_pct" in globals() and "sin_after_pct" in globals():
    insight_3 += f" Mejora frente al baseline: {sin_before_pct:.1f}% → {sin_after_pct:.1f}%."

# Normalizar variables para evitar errores de render
corte_txt = str(today.date()) if "today" in globals() else "última ejecución"
modelo_sent_txt = str(sentiment_model) if "sentiment_model" in globals() else "N/A"

top10_html_safe = top10_html if isinstance(top10_html, str) else "<p>No disponible.</p>"
terms_html_safe = terms_html if isinstance(terms_html, str) else "<p>No disponible.</p>"
metrics_html_safe = metrics_html if isinstance(metrics_html, str) else "<p>No disponible.</p>"

report_html = f"""<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="utf-8"/>
    <title>Reporte de Percepción de Seguridad - Cali</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 24px; color: #1f2937; }}
        h1, h2 {{ margin-bottom: 8px; }}
        .muted {{ color: #6b7280; }}
        .kpi-grid {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin: 14px 0 22px; }}
        .card {{ border: 1px solid #e5e7eb; border-radius: 10px; padding: 12px; background: #fff; }}
        .kpi {{ font-size: 28px; font-weight: 700; }}
        .lbl {{ font-size: 12px; color: #6b7280; }}
        .section {{ margin-top: 26px; }}
        table {{ border-collapse: collapse; width: 100%; font-size: 13px; }}
        th, td {{ border: 1px solid #e5e7eb; padding: 6px; text-align: left; }}
        th {{ background: #f9fafb; }}
        ul {{ margin-top: 6px; }}
    </style>
</head>
<body>
    <h1>Percepción de seguridad en Cali</h1>
    <p class="muted">Corte: {corte_txt} · Modelo sentimiento: {modelo_sent_txt}</p>

    <div class="kpi-grid">
        <div class="card"><div class="kpi">{total}</div><div class="lbl">Noticias analizadas (Cali)</div></div>
        <div class="card"><div class="kpi">{pct_neg:.1f}%</div><div class="lbl">Sentimiento negativo</div></div>
        <div class="card"><div class="kpi">{pct_alto_riesgo:.1f}%</div><div class="lbl">Alto riesgo (score ≥ 70)</div></div>
        <div class="card"><div class="kpi">{pct_sin_cls:.1f}%</div><div class="lbl">Sin clasificar (final)</div></div>
    </div>

    <div class="card">
        <b>Lectura ejecutiva:</b>
        <ul>
            <li>{insight_1}</li>
            <li>{insight_2}</li>
            <li>{insight_3}</li>
        </ul>
    </div>

    <div class="section">{plot_daily_html}</div>
    <div class="section">{plot_cat_html}</div>
    <div class="section">{plot_heat_html}</div>
    <div class="section">{plot_sent_src_html}</div>

    <div class="section card">
        <h2>Top 10 noticias con mayor riesgo</h2>
        {top10_html_safe}
    </div>

    <div class="section card">
        <h2>Términos más frecuentes en la conversación</h2>
        {terms_html_safe}
    </div>

    <div class="section card">
        <h2>Métricas de modelos (train/validación/predicción)</h2>
        {metrics_html_safe}
    </div>
</body>
</html>
"""

out_dir = Path("reports")
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / f"reporte_percepcion_seguridad_cali_{today.strftime('%Y%m%d') if 'today' in globals() else pd.Timestamp.now().strftime('%Y%m%d')}.html"

with open(out_file, "w", encoding="utf-8") as f:
    f.write(report_html)

print(f"Reporte HTML generado: {out_file.resolve()}")

Reporte HTML generado: A:\programing_workspace\SYJ_webScrapper\src\dataframe\reports\reporte_percepcion_seguridad_cali_20260312.html
